## Processor Modules Test

Day4 작업 중 아티클 처리 모듈 기능 테스트

테스트 대상:
- ArticleSummarizer: 아티클 요약 생성
- ImportanceEvaluator: 중요도 평가
- ContentClassifier: 콘텐츠 분류
- TextEmbedder: 텍스트 임베딩 생성
- ProcessingPipeline: 통합 처리 파이프라인

In [1]:
import asyncio
import json
import sys
from pathlib import Path

from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment variables
load_dotenv(project_root / ".env")

from app.processors.summarizer import ArticleSummarizer
from app.processors.evaluator import ImportanceEvaluator
from app.processors.classifier import ContentClassifier
from app.processors.embedder import TextEmbedder, get_embedder
from app.processors.pipeline import ProcessingPipeline, ProcessedArticle

print("✓ Setup complete")

✓ Setup complete


### Test Data Preparation

테스트에 사용할 샘플 아티클 준비

In [2]:
# 샘플 아티클 데이터
sample_paper = {
    "title": "Attention Is All You Need",
    "content": """
    The dominant sequence transduction models are based on complex recurrent or 
    convolutional neural networks in an encoder-decoder configuration. The best 
    performing models also connect the encoder and decoder through an attention 
    mechanism. We propose a new simple network architecture, the Transformer, 
    based solely on attention mechanisms, dispensing with recurrence and convolutions 
    entirely. Experiments on two machine translation tasks show these models to be 
    superior in quality while being more parallelizable and requiring significantly 
    less time to train.
    """,
    "url": "https://arxiv.org/abs/1706.03762",
    "source_name": "arXiv",
    "source_type": "paper",
    "metadata": {"year": 2017, "citations": 50000, "authors": ["Vaswani et al."]}
}

sample_news = {
    "title": "OpenAI Announces GPT-4 Turbo with Improved Performance",
    "content": """
    OpenAI today announced GPT-4 Turbo, the latest update to its flagship language model. 
    The new model offers improved performance, longer context windows, and reduced pricing. 
    GPT-4 Turbo features a 128K context window, knowledge up to April 2023, and better 
    instruction following. The model is now available through OpenAI's API at a fraction 
    of the cost of the original GPT-4.
    """,
    "url": "https://techcrunch.com/example",
    "source_name": "TechCrunch",
    "source_type": "news",
    "metadata": {"published_date": "2024-11-06"}
}

sample_articles = [sample_paper, sample_news]

print("Sample Articles Prepared:")
for i, article in enumerate(sample_articles, 1):
    print(f"  {i}. {article['title'][:50]}... ({article['source_type']})")

Sample Articles Prepared:
  1. Attention Is All You Need... (paper)
  2. OpenAI Announces GPT-4 Turbo with Improved Perform... (news)


### 1. ArticleSummarizer Tests

아티클 요약 생성 테스트

#### 1-1. Single Summarization (Korean, Medium)

In [3]:
# Summarizer 초기화
summarizer = ArticleSummarizer(provider="openai", temperature=0.3)

# 단일 요약 생성
summary_kr_medium = await summarizer.summarize(
    title=sample_paper["title"],
    content=sample_paper["content"],
    language="ko",
    length="medium"
)

print("Korean Medium Summary Test:")
print("=" * 60)
print(f"Title: {sample_paper['title']}")
print(f"\nSummary:")
print(summary_kr_medium)
print(f"\nSummary length: {len(summary_kr_medium)} characters")

Korean Medium Summary Test:
Title: Attention Is All You Need

Summary:
"Attention Is All You Need" 논문에서는 기존의 복잡한 순환 신경망이나 합성곱 신경망 기반의 시퀀스 변환 모델 대신, 주의 메커니즘만을 사용하는 새로운 네트워크 아키텍처인 Transformer를 제안합니다. 이 모델은 재귀와 합성곱을 완전히 배제하고, 주의 메커니즘을 통해 인코더와 디코더를 연결합니다. 두 가지 기계 번역 작업에 대한 실험 결과, Transformer 모델은 품질 면에서 우수하며, 병렬 처리 가능성이 높고 훈련 시간이 크게 단축된다는 것을 보여줍니다.

Summary length: 274 characters


#### 1-2. Different Summary Lengths Comparison

In [4]:
# 모든 길이 옵션 테스트
lengths = ["short", "medium", "long"]
summaries = {}

print("Summary Length Comparison:")
print("=" * 60)
print(f"Article: {sample_paper['title']}\n")

for length in lengths:
    summary = await summarizer.summarize(
        title=sample_paper["title"],
        content=sample_paper["content"],
        language="ko",
        length=length
    )
    summaries[length] = summary
    
    print(f"{length.upper()} ({len(summary)} chars):")
    print(f"  {summary}")
    print()

Summary Length Comparison:
Article: Attention Is All You Need

SHORT (185 chars):
  "Attention Is All You Need" 논문은 복잡한 순환 신경망이나 합성곱 신경망 대신, 전적으로 주의 메커니즘에 기반한 새로운 네트워크 아키텍처인 Transformer를 제안합니다. Transformer는 기계 번역 작업에서 더 높은 성능을 보이며, 병렬화가 용이하고 훈련 시간이 크게 단축된다는 장점을 입증했습니다.

MEDIUM (265 chars):
  "Attention Is All You Need" 논문에서는 기존의 복잡한 순환 신경망이나 합성곱 신경망 기반의 인코더-디코더 모델 대신, 주의 메커니즘만을 사용하는 새로운 네트워크 아키텍처인 트랜스포머를 제안합니다. 이 모델은 반복과 합성곱을 완전히 배제하고, 두 가지 기계 번역 작업에서 더 높은 품질을 제공하면서도 병렬 처리 가능성이 높고 훈련 시간이 크게 단축된다는 것을 실험을 통해 입증했습니다. 트랜스포머는 주의 메커니즘의 효율성을 극대화하여 성능을 향상시킵니다.

LONG (496 chars):
  "Attention Is All You Need" 논문은 기존의 복잡한 순차 변환 모델들이 주로 인코더-디코더 구조에서 복잡한 순환 신경망이나 합성곱 신경망을 기반으로 한다는 배경에서 출발합니다. 이러한 모델들은 주로 인코더와 디코더를 연결하는 주의 메커니즘을 사용하여 성능을 향상시킵니다. 이 논문에서는 주의 메커니즘만을 기반으로 하는 새로운 간단한 네트워크 아키텍처인 트랜스포머를 제안합니다. 트랜스포머는 순환이나 합성곱을 완전히 배제하고, 주의 메커니즘을 통해 인코더와 디코더를 연결합니다. 두 가지 기계 번역 작업에 대한 실험 결과, 트랜스포머 모델은 품질 면에서 우수할 뿐만 아니라 병렬 처리 가능성이 높고 훈련 시간이 크게 단축된다는 것을 보여줍니다. 이 연구는 주의 메커니즘의 중요성을 강조하며, 복잡한 신경망 구조 없이도 높은 성능을 달

#### 1-3. English vs Korean Summary

In [5]:
# 언어별 요약 비교
summary_kr = await summarizer.summarize(
    title=sample_news["title"],
    content=sample_news["content"],
    language="ko",
    length="short"
)

summary_en = await summarizer.summarize(
    title=sample_news["title"],
    content=sample_news["content"],
    language="en",
    length="short"
)

print("Language Comparison:")
print("=" * 60)
print(f"Article: {sample_news['title']}\n")
print(f"Korean Summary:")
print(f"  {summary_kr}")
print(f"\nEnglish Summary:")
print(f"  {summary_en}")

Language Comparison:
Article: OpenAI Announces GPT-4 Turbo with Improved Performance

Korean Summary:
  OpenAI가 GPT-4 Turbo를 발표했습니다. 이 모델은 128K 컨텍스트 윈도우, 2023년 4월까지의 지식, 향상된 명령어 수행 능력을 제공하며, 이전 GPT-4보다 비용이 절감된 형태로 OpenAI의 API를 통해 사용할 수 있습니다.

English Summary:
  OpenAI has introduced GPT-4 Turbo, an enhanced version of its language model featuring improved performance, a 128K context window, and better instruction following. The model, with knowledge up to April 2023, is available via OpenAI's API at a reduced cost compared to the original GPT-4.


#### 1-4. Batch Summarization

In [6]:
# 배치 요약
batch_summaries = await summarizer.batch_summarize(
    articles=sample_articles,
    language="ko",
    length="medium"
)

print("Batch Summarization Test:")
print("=" * 60)
print(f"Total articles: {len(sample_articles)}\n")

for i, (article, summary) in enumerate(zip(sample_articles, batch_summaries), 1):
    print(f"{i}. {article['title'][:50]}...")
    print(f"   Summary: {summary[:100]}...")
    print(f"   Length: {len(summary)} chars")
    print()

Batch Summarization Test:
Total articles: 2

1. Attention Is All You Need...
   Summary: "Attention Is All You Need" 논문은 기존의 복잡한 순환 신경망이나 합성곱 신경망 기반의 시퀀스 변환 모델 대신, 주의 메커니즘에만 의존하는 새로운 네트워크 아...
   Length: 286 chars

2. OpenAI Announces GPT-4 Turbo with Improved Perform...
   Summary: OpenAI가 최신 언어 모델인 GPT-4 Turbo를 발표했습니다. 이 모델은 성능 향상, 128K의 긴 컨텍스트 윈도우, 2023년 4월까지의 지식, 그리고 개선된 명령어 처리...
   Length: 183 chars



### 2. ImportanceEvaluator Tests

중요도 평가 테스트

#### 2-1. Single Article Evaluation

In [7]:
# Evaluator 초기화
evaluator = ImportanceEvaluator(provider="openai", temperature=0.2)

# 단일 평가
eval_result = await evaluator.evaluate(
    title=sample_paper["title"],
    content=sample_paper["content"],
    metadata=sample_paper["metadata"]
)

print("Importance Evaluation Test:")
print("=" * 60)
print(f"Article: {sample_paper['title']}\n")
print(f"Final Score: {eval_result['final_score']:.2f}")
print(f"  - LLM Score: {eval_result['llm_score']:.2f}")
print(f"  - Metadata Score: {eval_result['metadata_score']:.2f}")
print(f"\nDetailed Scores:")
print(f"  - Innovation: {eval_result['innovation']:.2f}")
print(f"  - Relevance: {eval_result['relevance']:.2f}")
print(f"  - Impact: {eval_result['impact']:.2f}")
print(f"  - Timeliness: {eval_result['timeliness']:.2f}")

if "reasoning" in eval_result:
    print(f"\nReasoning:")
    print(f"  {eval_result['reasoning']}")

Importance Evaluation Test:
Article: Attention Is All You Need

Final Score: 0.94
  - LLM Score: 1.00
  - Metadata Score: 0.80

Detailed Scores:
  - Innovation: 1.00
  - Relevance: 1.00
  - Impact: 1.00
  - Timeliness: 1.00

Reasoning:
  The paper 'Attention Is All You Need' introduces the Transformer model, which is a groundbreaking innovation in the field of AI and machine learning. It replaces traditional recurrent and convolutional neural networks with a novel architecture based solely on attention mechanisms. This approach significantly improves performance in sequence transduction tasks, such as machine translation, while also being more efficient in terms of parallelization and training time. The Transformer model has become a foundational architecture in AI, leading to numerous advancements and applications, including the development of large language models like BERT and GPT. Its relevance is extremely high as it aligns with current trends in AI research and has substantial pr

#### 2-2. Metadata Impact on Scores

In [8]:
# 메타데이터 유무에 따른 점수 비교
eval_without_metadata = await evaluator.evaluate(
    title=sample_paper["title"],
    content=sample_paper["content"],
    metadata={}
)

eval_with_metadata = await evaluator.evaluate(
    title=sample_paper["title"],
    content=sample_paper["content"],
    metadata=sample_paper["metadata"]
)

print("Metadata Impact Test:")
print("=" * 60)
print(f"Article: {sample_paper['title']}\n")
print(f"Without Metadata:")
print(f"  Final Score: {eval_without_metadata['final_score']:.2f}")
print(f"  Metadata Score: {eval_without_metadata['metadata_score']:.2f}")
print(f"\nWith Metadata (year=2017, citations=50000):")
print(f"  Final Score: {eval_with_metadata['final_score']:.2f}")
print(f"  Metadata Score: {eval_with_metadata['metadata_score']:.2f}")
print(f"\nScore Boost: {eval_with_metadata['final_score'] - eval_without_metadata['final_score']:.2f}")

Metadata Impact Test:
Article: Attention Is All You Need

Without Metadata:
  Final Score: 0.85
  Metadata Score: 0.50

With Metadata (year=2017, citations=50000):
  Final Score: 0.94
  Metadata Score: 0.80

Score Boost: 0.09


#### 2-3. Batch Evaluation

In [9]:
# 배치 평가
eval_results = await evaluator.batch_evaluate(sample_articles)

print("Batch Evaluation Test:")
print("=" * 60)
print(f"Total articles: {len(sample_articles)}\n")

for i, (article, result) in enumerate(zip(sample_articles, eval_results), 1):
    print(f"{i}. {article['title'][:50]}...")
    print(f"   Final Score: {result['final_score']:.2f}")
    print(f"   Innovation: {result['innovation']:.2f} | "
          f"Relevance: {result['relevance']:.2f} | "
          f"Impact: {result['impact']:.2f} | "
          f"Timeliness: {result['timeliness']:.2f}")
    print()

Batch Evaluation Test:
Total articles: 2

1. Attention Is All You Need...
   Final Score: 0.94
   Innovation: 1.00 | Relevance: 1.00 | Impact: 1.00 | Timeliness: 1.00

2. OpenAI Announces GPT-4 Turbo with Improved Perform...
   Final Score: 0.71
   Innovation: 0.60 | Relevance: 0.90 | Impact: 0.80 | Timeliness: 0.90



### 3. ContentClassifier Tests

콘텐츠 분류 테스트

#### 3-1. Single Article Classification

In [10]:
# Classifier 초기화
classifier = ContentClassifier(provider="openai", temperature=0.1)

# 단일 분류
class_result = await classifier.classify(
    title=sample_paper["title"],
    content=sample_paper["content"],
    source_name=sample_paper["source_name"],
    url=sample_paper["url"]
)

print("Content Classification Test:")
print("=" * 60)
print(f"Article: {sample_paper['title']}\n")
print(f"Category: {class_result['category']}")
print(f"Confidence: {class_result['confidence']:.2f}")
print(f"Research Field: {class_result['research_field']}")
print(f"Sub-fields: {', '.join(class_result['sub_fields'])}")
print(f"Keywords: {', '.join(class_result['keywords'][:5])}")

if class_result.get("reasoning"):
    print(f"\nReasoning:")
    print(f"  {class_result['reasoning']}")

Content Classification Test:
Article: Attention Is All You Need

Category: paper
Confidence: 1.00
Research Field: Machine Learning
Sub-fields: Natural Language Processing, Neural Networks
Keywords: Transformer, attention mechanism, sequence transduction, machine translation

Reasoning:
  The document is an academic paper published on arXiv, a well-known repository for research papers. The content discusses a novel neural network architecture, the Transformer, which is a significant contribution to the field of Machine Learning, specifically in Natural Language Processing. The presence of technical terms and experimental results further supports its classification as a research paper.


#### 3-2. Available Categories and Fields

In [11]:
# 사용 가능한 카테고리 및 연구 분야
print("Available Categories and Research Fields:")
print("=" * 60)

print(f"\nValid Categories:")
for cat in classifier.valid_categories:
    print(f"  - {cat}")

print(f"\nResearch Fields:")
for field in classifier.research_fields:
    print(f"  - {field}")

Available Categories and Research Fields:

Valid Categories:
  - paper
  - news
  - report
  - blog
  - other

Research Fields:
  - Machine Learning
  - Deep Learning
  - Natural Language Processing
  - Computer Vision
  - Reinforcement Learning
  - Robotics
  - AI Ethics
  - AI Infrastructure
  - Generative AI
  - Multimodal AI
  - Other


#### 3-3. Batch Classification

In [12]:
# 배치 분류
class_results = await classifier.batch_classify(sample_articles)

print("Batch Classification Test:")
print("=" * 60)
print(f"Total articles: {len(sample_articles)}\n")

for i, (article, result) in enumerate(zip(sample_articles, class_results), 1):
    print(f"{i}. {article['title'][:50]}...")
    print(f"   Category: {result['category']} (confidence: {result['confidence']:.2f})")
    print(f"   Research Field: {result['research_field']}")
    print(f"   Keywords: {', '.join(result['keywords'][:3])}")
    print()

Batch Classification Test:
Total articles: 2

1. Attention Is All You Need...
   Category: paper (confidence: 1.00)
   Research Field: Machine Learning
   Keywords: Transformer, attention mechanism, sequence transduction

2. OpenAI Announces GPT-4 Turbo with Improved Perform...
   Category: news (confidence: 0.90)
   Research Field: Natural Language Processing
   Keywords: GPT-4 Turbo, OpenAI, language model



#### 3-4. Category Distribution

In [13]:
# 카테고리 분포 계산
distribution = classifier.get_category_distribution(class_results)

print("Category Distribution:")
print("=" * 60)
for category, count in distribution.items():
    if count > 0:
        percentage = (count / len(sample_articles)) * 100
        print(f"  {category}: {count} ({percentage:.1f}%)")

Category Distribution:
  paper: 1 (50.0%)
  news: 1 (50.0%)


### 4. TextEmbedder Tests

텍스트 임베딩 생성 테스트

#### 4-1. Single Text Embedding

In [14]:
# Embedder 초기화
embedder = TextEmbedder(use_cache=True)

# 단일 임베딩 생성
text = sample_paper["title"]
embedding = await embedder.embed(text)

print("Single Text Embedding Test:")
print("=" * 60)
print(f"Text: {text}")
print(f"Embedding dimension: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")
print(f"Token count: {embedder.count_tokens(text)}")

Single Text Embedding Test:
Text: Attention Is All You Need
Embedding dimension: 1536
First 10 values: [0.046672966331243515, 0.00821229349821806, -0.024438994005322456, 0.038446538150310516, -0.04568353295326233, -0.05450361967086792, -0.00843844935297966, 0.06021406129002571, -0.030898576602339745, 0.02120213396847248]
Token count: 5


#### 4-2. Article Text Preparation and Embedding

In [15]:
# 아티클 텍스트 준비 및 임베딩
prepared_text = embedder.prepare_article_text(
    title=sample_paper["title"],
    content=sample_paper["content"],
    summary="Transformer 아키텍처를 제안하는 논문입니다."
)

article_embedding = await embedder.embed_article(
    title=sample_paper["title"],
    content=sample_paper["content"],
    summary="Transformer 아키텍처를 제안하는 논문입니다."
)

print("Article Embedding Test:")
print("=" * 60)
print(f"Article: {sample_paper['title']}\n")
print(f"Prepared Text Preview:")
print(f"  {prepared_text[:200]}...")
print(f"\nPrepared text tokens: {embedder.count_tokens(prepared_text)}")
print(f"Embedding dimension: {len(article_embedding)}")
print(f"First 5 values: {article_embedding[:5]}")

Article Embedding Test:
Article: Attention Is All You Need

Prepared Text Preview:
  Title: Attention Is All You Need

Summary: Transformer 아키텍처를 제안하는 논문입니다.

Content: 
    The dominant sequence transduction models are based on complex recurrent or 
    convolutional neural networks i...

Prepared text tokens: 140
Embedding dimension: 1536
First 5 values: [0.02123252861201763, -0.005046523176133633, -0.03452114388346672, -0.013761199079453945, 0.007173151709139347]


#### 4-3. Batch Embedding

In [16]:
# 배치 임베딩
texts = [article["title"] for article in sample_articles]
batch_embeddings = await embedder.batch_embed(texts, batch_size=5)

print("Batch Embedding Test:")
print("=" * 60)
print(f"Total texts: {len(texts)}\n")

for i, (text, embedding) in enumerate(zip(texts, batch_embeddings), 1):
    tokens = embedder.count_tokens(text)
    print(f"{i}. {text[:50]}...")
    print(f"   Tokens: {tokens} | Embedding dim: {len(embedding)}")
    print()

Batch Embedding Test:
Total texts: 2

1. Attention Is All You Need...
   Tokens: 5 | Embedding dim: 1536

2. OpenAI Announces GPT-4 Turbo with Improved Perform...
   Tokens: 12 | Embedding dim: 1536



#### 4-4. Embedding Cache Test

In [17]:
# 캐시 테스트
test_text = "Attention mechanism for neural networks"

# 첫 번째 호출
embedding1 = await embedder.embed(test_text)
cache_size_1 = embedder.get_cache_size()

# 두 번째 호출 (캐시 히트)
embedding2 = await embedder.embed(test_text)
cache_size_2 = embedder.get_cache_size()

print("Embedding Cache Test:")
print("=" * 60)
print(f"Text: {test_text}\n")
print(f"First call:")
print(f"  Cache size: {cache_size_1}")
print(f"  Embedding: {embedding1[:3]}...")
print(f"\nSecond call (should hit cache):")
print(f"  Cache size: {cache_size_2}")
print(f"  Embedding: {embedding2[:3]}...")
print(f"\nEmbeddings identical: {embedding1 == embedding2}")

# 캐시 통계
stats = embedder.get_cache_stats()
print(f"\nCache Stats: {stats}")

# 캐시 초기화
embedder.clear_cache()
print(f"\nCache cleared. New size: {embedder.get_cache_size()}")

Embedding Cache Test:
Text: Attention mechanism for neural networks

First call:
  Cache size: 4
  Embedding: [-0.008879438042640686, 0.017640307545661926, -0.018470285460352898]...

Second call (should hit cache):
  Cache size: 4
  Embedding: [-0.008879438042640686, 0.017640307545661926, -0.018470285460352898]...

Embeddings identical: True

Cache Stats: {'size': 4, 'enabled': True, 'model': 'text-embedding-3-small'}

Cache cleared. New size: 0


#### 4-5. Token Truncation Test

In [18]:
# 토큰 자르기 테스트
long_text = "AI research and development " * 1000  # Very long text

token_count_original = embedder.count_tokens(long_text)
print("Token Truncation Test:")
print("=" * 60)
print(f"Original text tokens: {token_count_original}")
print(f"Max tokens allowed: {embedder.MAX_TOKENS}")

# 자르기
truncated_text = embedder.truncate_text(long_text, max_tokens=1000)
token_count_truncated = embedder.count_tokens(truncated_text)

print(f"\nTruncated text tokens: {token_count_truncated}")
print(f"Truncation successful: {token_count_truncated <= 1000}")

# 임베딩 생성 (자동 자르기)
embedding_long = await embedder.embed(long_text, truncate=True)
print(f"\nEmbedding generated: {len(embedding_long)} dimensions")

Text truncated from 4001 to 1000 tokens


Token Truncation Test:
Original text tokens: 4001
Max tokens allowed: 8191

Truncated text tokens: 1000
Truncation successful: True

Embedding generated: 1536 dimensions


#### 4-6. Semantic Similarity Test

In [19]:
import numpy as np

# 코사인 유사도 계산
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 유사도 테스트용 텍스트
texts_for_similarity = [
    "Transformer architecture in deep learning",
    "Attention mechanism for neural networks",
    "Reinforcement learning for robotics",
    "Computer vision using CNNs"
]

similarity_embeddings = await embedder.batch_embed(texts_for_similarity)

print("Semantic Similarity Test:")
print("=" * 60)
print("Cosine Similarities:\n")

for i, text_i in enumerate(texts_for_similarity):
    for j, text_j in enumerate(texts_for_similarity):
        if i < j:
            similarity = cosine_similarity(
                similarity_embeddings[i],
                similarity_embeddings[j]
            )
            print(f"'{text_i}' vs")
            print(f"'{text_j}'")
            print(f"  Similarity: {similarity:.4f}\n")

Semantic Similarity Test:
Cosine Similarities:

'Transformer architecture in deep learning' vs
'Attention mechanism for neural networks'
  Similarity: 0.4854

'Transformer architecture in deep learning' vs
'Reinforcement learning for robotics'
  Similarity: 0.3040

'Transformer architecture in deep learning' vs
'Computer vision using CNNs'
  Similarity: 0.4224

'Attention mechanism for neural networks' vs
'Reinforcement learning for robotics'
  Similarity: 0.3289

'Attention mechanism for neural networks' vs
'Computer vision using CNNs'
  Similarity: 0.3612

'Reinforcement learning for robotics' vs
'Computer vision using CNNs'
  Similarity: 0.3520



### 5. ProcessingPipeline Tests

통합 처리 파이프라인 테스트

#### 5-1. Single Article Processing

In [21]:
# Pipeline 초기화
pipeline = ProcessingPipeline(
    provider="openai",
    summary_length="medium",
    summary_language="ko",
    use_embedding_cache=True
)

In [22]:
# 단일 아티클 전체 처리
processed = await pipeline.process_article(
    title=sample_paper["title"],
    content=sample_paper["content"],
    url=sample_paper["url"],
    source_name=sample_paper["source_name"],
    source_type=sample_paper["source_type"],
    metadata=sample_paper["metadata"]
)

print("Single Article Processing Test:")
print("=" * 60)
print(f"Title: {processed.title}")
print(f"\nSummary:")
print(f"  {processed.summary}")
print(f"\nScores:")
print(f"  Importance: {processed.importance_score:.2f}")
print(f"  Innovation: {processed.innovation_score:.2f}")
print(f"  Relevance: {processed.relevance_score:.2f}")
print(f"  Impact: {processed.impact_score:.2f}")
print(f"  Timeliness: {processed.timeliness_score:.2f}")
print(f"\nClassification:")
print(f"  Category: {processed.category}")
print(f"  Research Field: {processed.research_field}")
print(f"  Keywords: {', '.join(processed.keywords[:5])}")
print(f"\nEmbedding:")
print(f"  Dimension: {len(processed.embedding)}")
print(f"  First 5 values: {processed.embedding[:5]}")

Single Article Processing Test:
Title: Attention Is All You Need

Summary:
  "Attention Is All You Need" 논문은 기존의 복잡한 순환 신경망이나 합성곱 신경망 기반의 인코더-디코더 모델 대신, 주의 메커니즘만을 활용한 새로운 네트워크 아키텍처인 Transformer를 제안합니다. Transformer는 반복과 합성곱을 완전히 배제하고, 주의 메커니즘을 통해 인코더와 디코더를 연결합니다. 두 가지 기계 번역 작업에서 실험한 결과, Transformer 모델은 더 높은 품질을 제공하면서도 병렬화가 용이하고 학습 시간이 크게 단축됨을 보여주었습니다.

Scores:
  Importance: 0.94
  Innovation: 1.00
  Relevance: 1.00
  Impact: 1.00
  Timeliness: 1.00

Classification:
  Category: paper
  Research Field: Machine Learning
  Keywords: Transformer, attention mechanism, sequence transduction, machine translation

Embedding:
  Dimension: 1536
  First 5 values: [0.00540955038741231, 0.0020673158578574657, -0.07030940055847168, 0.003930998966097832, 0.019902896136045456]


### 5-2. Batch Processing

In [23]:
# 배치 처리
processed_batch = await pipeline.process_batch(
    articles=sample_articles,
    max_concurrent=2
)

print("Batch Processing Test:")
print("=" * 60)
print(f"Total articles processed: {len(processed_batch)}\n")

for i, article in enumerate(processed_batch, 1):
    print(f"{i}. {article.title[:50]}...")
    print(f"   Score: {article.importance_score:.2f} | "
          f"Category: {article.category} | "
          f"Field: {article.research_field}")
    print(f"   Summary: {article.summary[:100]}...")
    print()

Batch Processing Test:
Total articles processed: 2

1. Attention Is All You Need...
   Score: 0.94 | Category: paper | Field: Machine Learning
   Summary: "Attention Is All You Need"에서는 기존의 복잡한 순환 신경망이나 합성곱 신경망 기반의 인코더-디코더 모델 대신, 주의(attention) 메커니즘만을 사용하는...

2. OpenAI Announces GPT-4 Turbo with Improved Perform...
   Score: 0.71 | Category: news | Field: Natural Language Processing
   Summary: OpenAI가 GPT-4 Turbo를 발표했습니다. 이 최신 모델은 성능이 향상되고, 128K의 더 긴 컨텍스트 윈도우를 제공하며, 가격이 낮아졌습니다. 또한, 2023년 4월까지...



### 5-3. Top Articles Filtering

In [24]:
# 상위 아티클 추출
top_articles = pipeline.get_top_articles(processed_batch, top_n=2)

print("Top Articles Test:")
print("=" * 60)
print(f"Top {len(top_articles)} articles by importance:\n")

for i, article in enumerate(top_articles, 1):
    print(f"{i}. {article.title}")
    print(f"   Score: {article.importance_score:.2f}")
    print(f"   Category: {article.category}")
    print()

Top Articles Test:
Top 2 articles by importance:

1. Attention Is All You Need
   Score: 0.94
   Category: paper

2. OpenAI Announces GPT-4 Turbo with Improved Performance
   Score: 0.71
   Category: news



### 5-4. Category and Score Filtering

In [25]:
# 카테고리별 필터링
papers_only = pipeline.filter_by_category(processed_batch, category="paper")
news_only = pipeline.filter_by_category(processed_batch, category="news")

# 점수별 필터링
high_quality = pipeline.filter_by_score(processed_batch, min_score=0.7)

print("Filtering Test:")
print("=" * 60)
print(f"\nCategory Filtering:")
print(f"  Papers: {len(papers_only)}")
print(f"  News: {len(news_only)}")

print(f"\nScore Filtering (≥0.7):")
print(f"  High quality articles: {len(high_quality)}")

for article in high_quality:
    print(f"    - {article.title[:50]}... (score: {article.importance_score:.2f})")

Filtering Test:

Category Filtering:
  Papers: 1
  News: 1

Score Filtering (≥0.7):
  High quality articles: 2
    - Attention Is All You Need... (score: 0.94)
    - OpenAI Announces GPT-4 Turbo with Improved Perform... (score: 0.71)


### 5-5. Processing Statistics

In [26]:
# 처리 통계
stats = pipeline.get_statistics(processed_batch)

print("Processing Statistics:")
print("=" * 60)
print(f"Total articles: {stats['total']}")
print(f"\nScore Statistics:")
print(f"  Average: {stats['average_score']:.2f}")
print(f"  Maximum: {stats['max_score']:.2f}")
print(f"  Minimum: {stats['min_score']:.2f}")
print(f"  High quality (≥0.7): {stats['high_quality_count']}")
print(f"\nCategory Distribution:")
for category, count in stats['category_distribution'].items():
    percentage = (count / stats['total']) * 100
    print(f"  {category}: {count} ({percentage:.1f}%)")

Processing Statistics:
Total articles: 2

Score Statistics:
  Average: 0.82
  Maximum: 0.94
  Minimum: 0.71
  High quality (≥0.7): 2

Category Distribution:
  paper: 1 (50.0%)
  news: 1 (50.0%)


### 5-6. ProcessedArticle Serialization

In [27]:
# ProcessedArticle to dict 변환
article_dict = processed_batch[0].to_dict()

print("ProcessedArticle Serialization Test:")
print("=" * 60)
print(f"Article: {processed_batch[0].title}\n")
print("Serialized to dict:")
print(json.dumps({
    "title": article_dict["title"],
    "category": article_dict["category"],
    "importance_score": article_dict["importance_score"],
    "research_field": article_dict["research_field"],
    "keywords_count": len(article_dict["keywords"]),
    "embedding_dimension": len(article_dict["embedding"]),
    "processed_at": article_dict["processed_at"]
}, indent=2))

ProcessedArticle Serialization Test:
Article: Attention Is All You Need

Serialized to dict:
{
  "title": "Attention Is All You Need",
  "category": "paper",
  "importance_score": 0.94,
  "research_field": "Machine Learning",
  "keywords_count": 4,
  "embedding_dimension": 1536,
  "processed_at": "2025-12-15T10:18:34.115763"
}


### 6. Integration Test: End-to-End Workflow

실제 사용 시나리오 시뮬레이션

In [28]:
# 실제 워크플로우 시뮬레이션
async def simulate_daily_curation():
    """일일 큐레이션 파이프라인 시뮬레이션"""
    
    # 추가 샘플 아티클
    additional_articles = [
        {
            "title": "BERT: Pre-training of Deep Bidirectional Transformers",
            "content": "We introduce BERT, which stands for Bidirectional Encoder Representations from Transformers. BERT is designed to pre-train deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers.",
            "url": "https://arxiv.org/abs/1810.04805",
            "source_name": "arXiv",
            "source_type": "paper",
            "metadata": {"year": 2018, "citations": 30000}
        },
        {
            "title": "Google Announces Gemini 1.5 with 1M Context Window",
            "content": "Google has unveiled Gemini 1.5, featuring a breakthrough 1 million token context window. The model demonstrates exceptional long-context understanding and reasoning capabilities.",
            "url": "https://techcrunch.com/example-gemini",
            "source_name": "TechCrunch",
            "source_type": "news",
            "metadata": {"published_date": "2024-12-01"}
        },
    ]
    
    all_articles = sample_articles + additional_articles
    
    print("Daily Curation Workflow Simulation:")
    print("=" * 60)
    print(f"Total articles to process: {len(all_articles)}\n")
    
    # Step 1: 전체 처리
    print("Step 1: Processing all articles...")
    pipeline = ProcessingPipeline(
        provider="openai",
        summary_length="medium",
        summary_language="ko"
    )
    
    processed = await pipeline.process_batch(all_articles, max_concurrent=3)
    print(f"  ✓ Processed {len(processed)} articles\n")
    
    # Step 2: 고품질만 필터링 (점수 0.6 이상)
    print("Step 2: Filtering high-quality articles...")
    high_quality = pipeline.filter_by_score(processed, min_score=0.6)
    print(f"  ✓ Found {len(high_quality)} high-quality articles\n")
    
    # Step 3: 상위 3개 추출
    print("Step 3: Selecting top 3 articles...")
    top_3 = pipeline.get_top_articles(high_quality, top_n=3)
    print(f"  ✓ Selected {len(top_3)} top articles\n")
    
    # Step 4: 결과 출력
    print("=" * 60)
    print("Daily Digest - Top 3 Articles:")
    print("=" * 60)
    
    for i, article in enumerate(top_3, 1):
        print(f"\n{i}. {article.title}")
        print(f"   Score: {article.importance_score:.2f} | "
              f"Category: {article.category} | "
              f"Field: {article.research_field}")
        print(f"   Summary: {article.summary}")
        print(f"   Keywords: {', '.join(article.keywords[:5])}")
    
    # Step 5: 통계
    print("\n" + "=" * 60)
    print("Overall Statistics:")
    print("=" * 60)
    stats = pipeline.get_statistics(processed)
    print(f"Total processed: {stats['total']}")
    print(f"Average score: {stats['average_score']:.2f}")
    print(f"Category distribution: {stats['category_distribution']}")
    print(f"High quality articles: {stats['high_quality_count']}")

# 시뮬레이션 실행
await simulate_daily_curation()

Daily Curation Workflow Simulation:
Total articles to process: 4

Step 1: Processing all articles...
  ✓ Processed 4 articles

Step 2: Filtering high-quality articles...
  ✓ Found 4 high-quality articles

Step 3: Selecting top 3 articles...
  ✓ Selected 3 top articles

Daily Digest - Top 3 Articles:

1. Attention Is All You Need
   Score: 0.94 | Category: paper | Field: Machine Learning
   Summary: "Attention Is All You Need" 논문에서는 기존의 복잡한 순차 변환 모델들이 주로 순환 신경망이나 합성곱 신경망을 사용하여 인코더-디코더 구조를 형성하는 반면, 제안된 Transformer 모델은 전적으로 주의 메커니즘에 기반하여 이러한 복잡성을 제거합니다. 이 새로운 아키텍처는 반복과 합성곱을 완전히 배제하고, 두 개의 기계 번역 작업 실험에서 더 높은 품질을 제공하며 병렬 처리 가능성이 높고 훈련 시간이 크게 단축된다는 결과를 보여줍니다. Transformer는 주의 메커니즘만으로도 기존 모델을 능가할 수 있음을 입증합니다.
   Keywords: Transformer, attention mechanism, sequence transduction, machine translation

2. BERT: Pre-training of Deep Bidirectional Transformers
   Score: 0.90 | Category: paper | Field: Natural Language Processing
   Summary: BERT는 Bidirectional Encoder Representations from Transforme

### 7. Performance Benchmark

각 프로세서의 성능 측정

In [29]:
import time

# 성능 벤치마크
async def benchmark_processors():
    """각 프로세서 성능 측정"""
    
    test_article = sample_paper
    results = {}
    
    # 1. Summarizer
    summarizer = ArticleSummarizer(provider="openai")
    start = time.time()
    await summarizer.summarize(
        title=test_article["title"],
        content=test_article["content"],
        language="ko",
        length="medium"
    )
    results["Summarizer"] = time.time() - start
    
    # 2. Evaluator
    evaluator = ImportanceEvaluator(provider="openai")
    start = time.time()
    await evaluator.evaluate(
        title=test_article["title"],
        content=test_article["content"],
        metadata=test_article["metadata"]
    )
    results["Evaluator"] = time.time() - start
    
    # 3. Classifier
    classifier = ContentClassifier(provider="openai")
    start = time.time()
    await classifier.classify(
        title=test_article["title"],
        content=test_article["content"],
        source_name=test_article["source_name"],
        url=test_article["url"]
    )
    results["Classifier"] = time.time() - start
    
    # 4. Embedder
    embedder = TextEmbedder()
    start = time.time()
    await embedder.embed_article(
        title=test_article["title"],
        content=test_article["content"]
    )
    results["Embedder"] = time.time() - start
    
    # 5. Full Pipeline
    pipeline = ProcessingPipeline(provider="openai")
    start = time.time()
    await pipeline.process_article(
        title=test_article["title"],
        content=test_article["content"],
        url=test_article["url"],
        source_name=test_article["source_name"],
        metadata=test_article["metadata"]
    )
    results["Full Pipeline"] = time.time() - start
    
    return results

# 벤치마크 실행
print("Performance Benchmark:")
print("=" * 60)
benchmark_results = await benchmark_processors()

for processor, elapsed in benchmark_results.items():
    print(f"{processor:20s}: {elapsed:.2f}s")

print("\n" + "=" * 60)
print("Note: Full Pipeline processes all steps in parallel,")
print("so it's faster than running each processor sequentially.")

Performance Benchmark:
Summarizer          : 1.91s
Evaluator           : 3.19s
Classifier          : 2.06s
Embedder            : 0.24s
Full Pipeline       : 4.05s

Note: Full Pipeline processes all steps in parallel,
so it's faster than running each processor sequentially.


### Summary

이 노트북에서 다룬 내용:

#### 1. ArticleSummarizer (요약 생성)
- ✓ 단일 요약 생성 (Korean/English)
- ✓ 요약 길이 비교 (short/medium/long)
- ✓ 언어별 요약 비교
- ✓ 배치 요약

#### 2. ImportanceEvaluator (중요도 평가)
- ✓ 단일 아티클 평가
- ✓ 메타데이터 영향 분석
- ✓ 배치 평가
- ✓ 4가지 평가 기준 (Innovation, Relevance, Impact, Timeliness)

#### 3. ContentClassifier (콘텐츠 분류)
- ✓ 단일 분류 (paper/news/report/blog/other)
- ✓ 연구 분야 및 키워드 추출
- ✓ 배치 분류
- ✓ 카테고리 분포 계산

#### 4. TextEmbedder (임베딩 생성)
- ✓ 단일 텍스트 임베딩
- ✓ 아티클 임베딩 (제목+내용+요약)
- ✓ 배치 임베딩
- ✓ 캐싱 기능
- ✓ 토큰 자르기
- ✓ 시맨틱 유사도 계산

#### 5. ProcessingPipeline (통합 파이프라인)
- ✓ 단일 아티클 전체 처리
- ✓ 배치 처리 (병렬 실행)
- ✓ 상위 아티클 추출
- ✓ 카테고리/점수별 필터링
- ✓ 처리 통계
- ✓ ProcessedArticle 직렬화

#### 6. Integration Tests
- ✓ End-to-End 워크플로우 시뮬레이션
- ✓ 성능 벤치마크

모든 프로세서 모듈이 정상적으로 작동